# LOGAN: contig taxonomy and the eukaryotic test

**Draft caption.** Kraken2 (2.17.2; PlusPFP 2023-06-05) classification of the 79,774 contigs carrying the Foldseek queries, re-streamed from LOGAN. (a) Contig class for each arm. (b) For candidates and complete Prodigal genes whose best full-length target is eukaryotic, the class of their contig: 69.8% of such candidates (n = 2,525) lie on eukaryotic contigs. PlusPFP contains a single animal genome (human), so animal contigs are reported as one group; 15.6% of contigs are unclassified.

**Outputs:** `../images/figL08_contig_taxonomy.pdf` (vector) and `../images/figL08_contig_taxonomy.png` (300 dpi).

**Data** (plotting-ready, committed; produced by `../prep/logan_prepare_data.py` on Setonix):

- `../data/figL08_contig_group_by_arm.tsv.gz` (4 rows; sha256 `9b29a817b922…`): Kraken2 (PlusPFP) class of each query's contig, share by arm. Source on Setonix (`$LOGAN_ROOT`): `qc/candidates/taxonomy/contig_group_by_arm.tsv`.
- `../data/figL08_hit_vs_contig.tsv.gz` (103 rows; sha256 `9f07f8535555…`): queries with a full-length hit: best-target group x contig group. Source on Setonix (`$LOGAN_ROOT`): `qc/candidates/taxonomy/hit_vs_contig.tsv`.

**Upstream.** LOGAN v1.2 metagenomes (50,009), genome_entropy #102 (closed by PR #115); analysis scripts `Validation/logan/scripts/` at commit `06080b4`. Full run provenance: `../data/logan_provenance.json`.

**Run:** `jupyter nbconvert --to notebook --execute figL08_contig_taxonomy.ipynb` from `manuscript/notebooks/`. Needs only pandas, numpy, matplotlib and seaborn; no HPC or remote paths.

In [ ]:
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

DATA, IMAGES = Path("../data"), Path("../images")
IMAGES.mkdir(exist_ok=True)
plt.rcParams["svg.hashsalt"] = "genome_entropy"   # deterministic SVG ids if SVG is ever written

# Colour slots (fixed order, colour-blind-checked categorical palette used across the LOGAN figures)
SLOTS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7"]
INK, INK2, GRID, SURFACE = "#0b0b0b", "#52514e", "#e4e3df", "#fcfcfb"
sns.set_theme(context="paper", style="ticks", font_scale=1.15, rc={
    "axes.edgecolor": INK2, "axes.labelcolor": INK, "xtick.color": INK2, "ytick.color": INK2,
    "text.color": INK, "axes.facecolor": SURFACE, "figure.facecolor": "white", "grid.color": GRID,
    "axes.grid": True, "grid.linewidth": 0.6, "axes.spines.top": False, "axes.spines.right": False,
    "legend.frameon": False, "lines.linewidth": 2, "font.family": "DejaVu Sans"})


def panel(ax, letter, x=-0.08, y=1.04):
    ax.text(x, y, letter, transform=ax.transAxes, fontsize=15, fontweight="bold", va="bottom", ha="right")


def save(fig, name):
    """Vector PDF (fixed metadata, so reruns are identical) and a 300-dpi PNG preview."""
    fig.savefig(IMAGES / f"{name}.pdf", bbox_inches="tight", metadata={"CreationDate": None, "ModDate": None})
    fig.savefig(IMAGES / f"{name}.png", dpi=300, bbox_inches="tight", metadata={"Software": None})
    plt.close(fig)
    print("wrote", IMAGES / f"{name}.pdf", IMAGES / f"{name}.png")

In [ ]:
C = pd.read_csv(DATA / "figL08_contig_group_by_arm.tsv.gz", sep="\t", index_col=0)
X = pd.read_csv(DATA / "figL08_hit_vs_contig.tsv.gz", sep="\t")
GROUPS = ["Bacteria", "Archaea", "Viruses", "plants", "animals", "fungi", "other eukaryotes", "unclassified"]
GC = dict(zip(GROUPS, [SLOTS[0], SLOTS[6], SLOTS[1], SLOTS[5], SLOTS[4], SLOTS[3], SLOTS[2], GRID]))
EUK = {"plants", "animals", "fungi", "other eukaryotes"}
ARMS = ["prodigal", "candidate", "intergenic_lo", "antisense"]
LABEL = {"prodigal": "complete Prodigal genes", "candidate": "candidates", "intergenic_lo": "low-score intergenic",
         "antisense": "antisense shadows"}

fig, axes = plt.subplots(1, 2, figsize=(16, 4.8), gridspec_kw={"width_ratios": [1.3, 1]})
ax = axes[0]
Cc = C.reindex(ARMS)[GROUPS]
left = np.zeros(len(ARMS))
for g in GROUPS:
    ax.barh(range(len(ARMS)), Cc[g], left=left, color=GC[g], label=g, height=0.6, edgecolor="white", lw=2)
    left += Cc[g].to_numpy()
ax.set_yticks(range(len(ARMS)), [LABEL[a] for a in ARMS])
ax.invert_yaxis()
ax.set_xlim(0, 1)
ax.set_xlabel("share of queries, by Kraken2 class of the contig")
ax.legend(ncol=4, loc="lower center", bbox_to_anchor=(0.45, 1.0), fontsize=9)
panel(ax, "a", x=-0.32)
ax = axes[1]
arms = ["prodigal", "candidate"]
e = X[X.arm.isin(arms) & X.hit_group.isin(EUK)]
S = e.pivot_table(index="arm", columns="contig_group", values="queries", aggfunc="sum", fill_value=0)
n = S.sum(axis=1)
S = S.div(n, axis=0).reindex(index=arms, columns=GROUPS, fill_value=0)
left = np.zeros(len(arms))
for g in GROUPS:
    ax.barh(range(len(arms)), S[g], left=left, color=GC[g], height=0.6, edgecolor="white", lw=2)
    left += S[g].to_numpy()
ax.set_yticks(range(len(arms)), [f"{LABEL[a]}\n(n = {int(n[a]):,})" for a in arms])
ax.invert_yaxis()
ax.set_xlim(0, 1)
ax.set_xlabel("queries with a eukaryotic best hit: Kraken2 class of the contig")
panel(ax, "b", x=-0.3)
fig.tight_layout()
save(fig, "figL08_contig_taxonomy")